# Exploratory Data Analysis (EDA) for Givemore

Import Libraries

In [46]:
import pandas as pd
import numpy as np

Load Data

In [37]:
ratings = pd.read_csv("/home/bukunmi/code/givemore/data/ratings.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [38]:
movies = pd.read_csv("/home/bukunmi/code/givemore/data/movies.csv")
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


### **1.** What's the exact density of the user-movie matrix, and what does it imply for the minimum number of co-ratings before a similarity is meaningful?

  $$\text{density} = \frac{\text{ratings}}{\text{users} \times \text{movies}}$$

In [39]:
num_ratings = ratings["rating"].count()
print(f"Number of ratings: {num_ratings}")
num_users = ratings["userId"].nunique()
print(f"Number of users: {num_users}")
num_movies = movies["movieId"].nunique()
print(f"Number of movies: {num_movies}")

Number of ratings: 100836
Number of users: 610
Number of movies: 9742


In [40]:
density = num_ratings / (num_users * num_movies)
print(f"Density: {(density*100):.2f}%")

Density: 1.70%


At **_1.70%_** density the matrix is **_98.3%_** empty. The average user has rated 165 of 9,742 movies, and typical movie pairs share 0 raters. Behavioural (collaborative) similarity is therefore only reliable for the popular head; the long tail needs a content-based fallback and a minimum co-rating threshold to avoid spurious similarities.

In [54]:
movie_ratings_counts = ratings.groupby("movieId")["userId"].count()
movie_ratings_counts.describe()

count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       329.000000
Name: userId, dtype: float64

In [ ]:
# Predict expected overlap BEFORE measuring (independence assumption):
#   E[overlap] = n_A * n_B / U
U = num_users
med = movie_ratings_counts.median()   # typical movie
mx = movie_ratings_counts.max()       # blockbuster

print(f"predicted overlap, median x median ({med:.0f}x{med:.0f}): {med*med/U:.3f}  -> ~nobody")
print(f"predicted overlap, max x max ({mx:.0f}x{mx:.0f}):       {mx*mx/U:.1f}    -> lots")

In [42]:
thresholds = [2, 5, 10, 20, 50]
for t in thresholds:
    movies_with_enough_ratings = (movie_ratings_counts >= t).sum()
    print(f"Threshold: {t}, Movies with enough ratings: {movies_with_enough_ratings}")

Threshold: 2, Movies with enough ratings: 6278
Threshold: 5, Movies with enough ratings: 3650
Threshold: 10, Movies with enough ratings: 2269
Threshold: 20, Movies with enough ratings: 1297
Threshold: 50, Movies with enough ratings: 450


**_3446_** movies have less than **2** ratings

In [43]:
user_movie = ratings.assign(watched=1).pivot_table(
    index="userId",
    columns="movieId",
    values="watched",
    fill_value=0
)
user_movie

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
607,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
608,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [49]:
co_ratings = (user_movie.T @ user_movie).to_numpy().copy()
np.fill_diagonal(co_ratings, 0)

In [50]:
for t in thresholds:
    pair_count = (co_ratings >= t).sum().sum() // 2
    print(f"Movie pairs with at least {t} co-ratings: {pair_count:,}")

Movie pairs with at least 2 co-ratings: 4,738,640
Movie pairs with at least 5 co-ratings: 1,293,963
Movie pairs with at least 10 co-ratings: 434,435
Movie pairs with at least 20 co-ratings: 113,136
Movie pairs with at least 50 co-ratings: 9,912


The number of possible movie pairs is:
$$\frac{9724 \times 9723} {2}$$
which is 47,273,226 possible movie pairs

Although millions of movie pairs have at least 2 shared raters, this is still only a small fraction of all possible movie pairs. The drop-off is steep as the threshold increases: 5 co-ratings leaves about 1.29M pairs, 10 leaves about 434K, 20 leaves about 113K, and 50 leaves only about 9.9K.

The connected pairs exist, but they're concentrated among popular movies which is precisely why the long tail still needs a content fallback.

### Q1 Decision

**Density is 1.70% (98.3% empty).** The prediction held: typical movie pairs share **0** raters, while only blockbusters share many and the measured distribution confirms **90%** of all pairs share 0–1 raters, with connected pairs concentrated in the popular head.

**Decision:** require **>= 5 co-ratings** before trusting a collaborative similarity (keeps 1.29M pairs ≈ 2.7% of all possible). Below that threshold, fall back to **content similarity** so long-tail movies remain recommendable.

### **2.** How many movie *pairs* actually share enough common raters to compute a stable similarity?

Answered by Q1's co-rating analysis **1,293,963 pairs** clear the c=5 threshold

### **3.** How many movies are effectively un-recommendable because they have too few ratings to ever appear as a neighbour?

Based on the decision in Q1, we would require >= 5 co-ratings before trusting collaborative similarity. Movies with fewer that 5 ratings will be considered unrecommendable by cs.

In [59]:
min_common_users = 5

# including movies not rated at all
movie_ratings_counts_inc_na = (
    ratings.groupby("movieId")["userId"]
    .count()
    .reindex(movies["movieId"], fill_value=0)
)

unrecommendable_movies = movie_ratings_counts_inc_na[movie_ratings_counts_inc_na < min_common_users]
len(unrecommendable_movies)

6092

In [63]:
have_enough_ratings = int((movie_ratings_counts >= min_common_users).sum())
has_stable_neighbor = (co_ratings >= min_common_users).any(axis=1)
recommendable = int(has_stable_neighbor.sum())
never_rated = movies["movieId"].nunique() - co_ratings.shape[0]
unrecommendable_exact = (co_ratings.shape[0] - recommendable) + never_rated

print(f"recommendable (>=1 neighbour with >=5 co-raters): {recommendable}")
print(f"have >=5 ratings but NO stable neighbour: {have_enough_ratings - recommendable}")
print(f"exact un-recommendable: {unrecommendable_exact} ({unrecommendable_exact / movies['movieId'].nunique() * 100:.1f}% of catalogue)")

recommendable (>=1 neighbour with >=5 co-raters): 3634
have >=5 ratings but NO stable neighbour: 16
exact un-recommendable: 6108 (62.7% of catalogue)


**Lower bound (Layer 1):** Any movie with fewer than 5 ratings can never reach 5 shared raters with another movie (co-ratings <= min of the two movies' counts). That's **6092** movies guaranteed un-recommendable by collaborative filtering — 6074 with too few ratings, plus 18 never rated at all.

**Exact answer (Layer 2):** Having ≥ 5 ratings is *necessary* but not *sufficient* — a movie also needs one other movie its raters actually overlap with. Of the 3650 movies with ≥ 5 ratings, only **3634** have a stable neighbour; the other **16** have enough ratings but their raters are too scattered to overlap with any single film. So the exact figure is **6108 un-recommendable (62.7% of the catalogue)** — only **3634 movies (37.3%) are reachable by item-item collaborative filtering at all**.

The Layer-1 bound was nearly tight here (6092 vs 6108) — only 16 movies fall in the gap — so for this dataset "too few ratings" is almost the whole story.

**Implication:** these 6100 movies are not unrecommendable *in general* — they just need a non-collaborative path: content (genre/title) similarity or popularity fallback. With 63% of the catalogue CF-invisible, the content path is carrying the majority of the long tail, not just patching edge cases.

# ==============================================================================